In [ ]:
import os
import json
import time
import pandas as pd
from pydantic import BaseModel, model_validator, ValidationError
from google import genai
from google.genai import types, errors

# Initialize the unified Google GenAI client
# It automatically picks up the GEMINI_API_KEY environment variable.
client = genai.Client()



In [ ]:
# ==========================================
# 1. SYMBOLIC LAYER (PYDANTIC DEFINITION)
# ==========================================
class InvoiceSummary(BaseModel):
    total_net_worth: float
    total_vat: float
    total_gross_worth: float

class Invoice(BaseModel):
    invoice_no: str
    client_name: str
    summary: InvoiceSummary

    @model_validator(mode='after')
    def check_mathematical_integrity(self):
        """Symbolic rule: Net Worth + VAT must exactly equal Gross Worth."""
        expected_gross = round(self.summary.total_net_worth + self.summary.total_vat, 2)
        if round(self.summary.total_gross_worth, 2) != expected_gross:
            raise ValueError(
                f"Math Failure: Net ({self.summary.total_net_worth}) + "
                f"VAT ({self.summary.total_vat}) != Gross ({self.summary.total_gross_worth})"
            )
        return self

x

In [ ]:
# ==========================================
# 2. NEURAL LAYER (LIVE GEMINI API INTEGRATION)
# ==========================================
def extract_invoice_data(text_content: str, max_retries=3):
    """
    Executes a live LLM call instructing Gemini to output strict JSON.
    Includes exponential backoff for rate limit handling in batch jobs.
    """
    prompt = f"""
    Extract the following invoice data into a strict JSON format.
    Required keys: "invoice_no" (string), "client_name" (string), 
    and a nested "summary" object containing "total_net_worth" (float), 
    "total_vat" (float), and "total_gross_worth" (float).
    
    Invoice Text:
    {text_content}
    """
    
    for attempt in range(max_retries):
        try:
            # Using 2.5-flash for the optimal balance of speed and reasoning in batch extraction
            response = client.models.generate_content(
                model='gemini-2.5-flash', 
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.1 # Low temperature for extraction tasks
                )
            )
            return response.text
        
        except errors.APIError as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt) 
            else:
                print(f"API Error after {max_retries} attempts: {e}")
                return None
        except Exception as e:
            print(f"Unexpected Error: {e}")
            return None



In [ ]:
# ==========================================
# 3. EXECUTION AND DUAL-PARADIGM EVALUATION
# ==========================================
def run_live_evaluation(csv_path='batch_1.csv', sample_size=100):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Dataset {csv_path} not found in the directory.")
        
    df = pd.read_csv(csv_path).head(sample_size)
    
    metrics = {
        "pure_llm": {"success": 0, "syntax_errors": 0, "silent_math_errors": 0},
        "neuro_symbolic": {"success": 0, "syntax_errors_caught": 0, "logic_errors_caught": 0}
    }

    print(f"Starting live evaluation of {len(df)} invoices via Gemini API...")
    
    for index, row in df.iterrows():
        if index % 10 == 0 and index > 0:
            print(f"Processed {index}/{len(df)}...")
            
        # 1. Neural Extraction (Single API Call per invoice)
        raw_json_string = extract_invoice_data(row['ocred_text'])
        
        if not raw_json_string:
            # API completely failed to return a string
            metrics["pure_llm"]["syntax_errors"] += 1
            metrics["neuro_symbolic"]["syntax_errors_caught"] += 1
            continue

        # 2. Evaluate Pure LLM Paradigm (Blind Trust)
        try:
            parsed = json.loads(raw_json_string)
            # A pure LLM pipeline takes the data as-is. We test if the logic holds.
            net = float(parsed['summary']['total_net_worth'])
            vat = float(parsed['summary']['total_vat'])
            gross = float(parsed['summary']['total_gross_worth'])
            
            if round(net + vat, 2) != round(gross, 2):
                metrics["pure_llm"]["silent_math_errors"] += 1
            else:
                metrics["pure_llm"]["success"] += 1
        except (json.JSONDecodeError, KeyError, ValueError, TypeError):
            # Fails if JSON is malformed or missing the required schema keys
            metrics["pure_llm"]["syntax_errors"] += 1

        # 3. Evaluate Hybrid Neuro-Symbolic Paradigm (Pydantic Engine)
        try:
            parsed = json.loads(raw_json_string)
            # The Symbolic rules engine enforces strict typing and mathematical integrity
            validated_invoice = Invoice(**parsed) 
            metrics["neuro_symbolic"]["success"] += 1
        except json.JSONDecodeError:
            metrics["neuro_symbolic"]["syntax_errors_caught"] += 1
        except ValidationError as e:
            # Pydantic successfully intercepted the LLM's hallucination or structural drift
            metrics["neuro_symbolic"]["logic_errors_caught"] += 1
            
        # Optional: Sleep to respect rate limits if using a free-tier API key
        time.sleep(1)

    return metrics

# Run the execution
if __name__ == "__main__":
    start_time = time.time()
    results = run_live_evaluation(sample_size=100)
    duration = round(time.time() - start_time, 2)
    
    print("\n==================================================")
    print("     LIVE API EVALUATION RESULTS (GEMINI)         ")
    print("==================================================")
    print(f"Sample Size: 100 Invoices | Execution Time: {duration}s\n")
    
    print("[BASELINE] PURE NEURAL ARCHITECTURE:")
    print(f"   - Successful Extractions:  {results['pure_llm']['success']}")
    print(f"   - Structural/Key Failures: {results['pure_llm']['syntax_errors']}")
    print(f"   - Silent Math Anomalies:   {results['pure_llm']['silent_math_errors']} (Data Poisoning!)\n")

    print("[PROPOSED] HYBRID NEURO-SYMBOLIC ARCHITECTURE:")
    print(f"   - Validated Extractions:   {results['neuro_symbolic']['success']}")
    print(f"   - Structural Drifts Blocked:{results['neuro_symbolic']['syntax_errors_caught']}")
    print(f"   - Math Hallucinations Blocked: {results['neuro_symbolic']['logic_errors_caught']}")
    print("==================================================")